# From Forecast to Recommendation

## Writing the memo that tells the grid operator how much battery capacity to build

The explainer drew a clear line between three levels of analytics. Descriptive: what happened. Predictive: what will happen. Prescriptive: what should we do. Each level inherits the uncertainty of the one below it, and each adds its own assumptions. The explainer also warned that the most common communication failure is mixing these levels without making the boundaries visible.

This notebook crosses all three levels deliberately. We will take the forecasting work from the previous notebook and push it to its logical conclusion: a stakeholder briefing that sizes battery storage for the grid. Along the way we will calculate the supply gap, identify critical hours, run scenario analyses, and write a recommendation that clearly separates observation from prediction from prescription.

The grid operator does not want a chart. They want an answer: how much battery storage should we build?

In [ ]:
%pip install -q pandas matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)

In [ ]:
demand = pd.read_csv("../data/grid_demand_hourly.csv", parse_dates=["timestamp"], index_col="timestamp")
solar = pd.read_csv("../data/solar_output.csv", parse_dates=["timestamp"], index_col="timestamp")
wind = pd.read_csv("../data/wind_output.csv", parse_dates=["timestamp"], index_col="timestamp")

---

## 1. Building the combined hourly picture

The grid operator thinks in terms of total renewable supply versus total demand. In the first notebook we saw these series separately; now we combine them into a single metric. The **supply gap** — demand minus renewable generation — is the number the operator needs to plan around.

In [ ]:
hourly = pd.DataFrame({
    "demand": demand["demand_mw"],
    "solar": solar["output_mw"],
    "wind": wind["output_mw"],
})

hourly["renewable_total"] = hourly["solar"] + hourly["wind"]
hourly["supply_gap"] = hourly["demand"] - hourly["renewable_total"]

hourly.head(10)

The **supply gap** is the amount of demand that renewables cannot cover. When the gap is positive, the grid needs backup generation (gas, nuclear, imports, or battery discharge). When negative, renewables exceed demand and the surplus can charge batteries or be exported. This is purely descriptive so far — we are reporting what happened, not predicting what will happen.

In [ ]:
# Daily averages
daily = hourly.resample("D").mean()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(daily.index, daily["demand"], label="Demand", linewidth=1.5)
ax.plot(daily.index, daily["renewable_total"], label="Renewable total", linewidth=1.5)
ax.fill_between(daily.index, daily["renewable_total"], daily["demand"],
                where=daily["demand"] > daily["renewable_total"],
                alpha=0.3, color="red", label="Supply gap")
ax.fill_between(daily.index, daily["renewable_total"], daily["demand"],
                where=daily["demand"] <= daily["renewable_total"],
                alpha=0.3, color="green", label="Surplus")
ax.set_title("Daily Demand vs Renewable Generation")
ax.set_ylabel("MW")
ax.legend()
plt.tight_layout()
plt.show()

The red shading shows the supply gap: months where demand exceeds renewable generation. The green shows the surplus. The seasonal mismatch we identified in notebook 1 is visible here as a planning problem: the gap is largest in winter (high demand, low solar) and the surplus appears in summer (low demand, high solar and wind).

---

## 2. Forecasting the supply gap

Now we move from description to prediction. We will build rolling average forecasts for demand, solar, and wind separately, then combine them to forecast the supply gap. This is the same rolling average technique from notebook 2, applied to all three series at once.

In [ ]:
# 7-day rolling average forecasts (shifted by 1 day to avoid lookahead)
daily["demand_forecast"] = daily["demand"].rolling(7).mean().shift(1)
daily["solar_forecast"] = daily["solar"].rolling(7).mean().shift(1)
daily["wind_forecast"] = daily["wind"].rolling(7).mean().shift(1)

daily["renewable_forecast"] = daily["solar_forecast"] + daily["wind_forecast"]
daily["gap_forecast"] = daily["demand_forecast"] - daily["renewable_forecast"]

daily[["supply_gap", "gap_forecast"]].dropna().tail(10)

In [ ]:
# December comparison
dec = daily.loc["2024-12"]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(dec.index, dec["supply_gap"], label="Actual gap", marker="o", markersize=4)
ax.plot(dec.index, dec["gap_forecast"], label="Forecast gap", marker="s", markersize=4)
ax.axhline(y=0, color="grey", linestyle="--", alpha=0.5)
ax.set_title("Supply Gap: Actual vs Forecast (December 2024)")
ax.set_ylabel("MW")
ax.legend()
plt.tight_layout()
plt.show()

---

## 3. Identifying critical hours

A forecast of the average gap is useful, but the grid operator also needs to know about the extremes. How often does the supply gap exceed a dangerous threshold? When do those critical hours cluster? This analysis moves us closer to a concrete engineering question.

In [ ]:
# Define a threshold: hours where the supply gap exceeds 30,000 MW
threshold = 30000

critical_hours = hourly[hourly["supply_gap"] > threshold]
total_hours = len(hourly)

print(f"Total hours in 2024:       {total_hours:,}")
print(f"Critical hours (>{threshold:,} MW):  {len(critical_hours):,}")
print(f"Percentage:                {len(critical_hours) / total_hours * 100:.1f}%")

In [ ]:
# When do critical hours occur? Group by month
critical_by_month = critical_hours.groupby(critical_hours.index.month).size()
critical_by_month.index = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                           "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"][:len(critical_by_month)]

critical_by_month.plot(kind="bar", title=f"Critical Hours by Month (gap > {threshold:,} MW)",
                       ylabel="Hours", figsize=(10, 5))
plt.tight_layout()
plt.show()

In [ ]:
# What time of day are critical hours most common?
critical_by_hour = critical_hours.groupby(critical_hours.index.hour).size()

critical_by_hour.plot(kind="bar", title="Critical Hours by Time of Day",
                      xlabel="Hour", ylabel="Count", figsize=(10, 5))
plt.tight_layout()
plt.show()

Critical hours cluster in winter evenings: high demand (heating, lighting) combined with zero solar output. This is precisely when battery storage would need to discharge. The pattern confirms what the seasonal and daily analysis in notebook 1 suggested — the timing mismatch between renewable supply and peak demand is the core problem.

---

## 4. Calculating required battery storage

Battery storage is measured in **MWh** (megawatt-hours). If the supply gap is 35,000 MW for one hour, we need 35,000 MWh of stored energy to cover that hour. Consecutive critical hours require cumulative storage — and it is the worst consecutive run, not the average, that determines the minimum battery size.

This is where the analysis starts to generate a number that can inform a recommendation.

In [ ]:
# Calculate how much energy (MWh) is needed above the threshold each hour
hourly["shortfall_mwh"] = (hourly["supply_gap"] - threshold).clip(lower=0)

# Find consecutive runs of shortfall
hourly["is_critical"] = hourly["shortfall_mwh"] > 0
hourly["critical_block"] = (hourly["is_critical"] != hourly["is_critical"].shift()).cumsum()

# For each consecutive block of critical hours, sum the shortfall
critical_blocks = (hourly[hourly["is_critical"]]
                   .groupby("critical_block")["shortfall_mwh"]
                   .agg(["sum", "count"]))
critical_blocks.columns = ["total_mwh", "duration_hours"]
critical_blocks = critical_blocks.sort_values("total_mwh", ascending=False)

print("Top 10 longest/largest shortfall events:")
critical_blocks.head(10)

In [ ]:
worst_case_mwh = critical_blocks["total_mwh"].max()
worst_case_hours = critical_blocks.loc[critical_blocks["total_mwh"].idxmax(), "duration_hours"]

print(f"Worst-case event: {worst_case_mwh:,.0f} MWh over {worst_case_hours} consecutive hours")
print(f"That is equivalent to {worst_case_mwh / 1000:,.1f} GWh of battery capacity")

---

## 5. Scenario analysis

The numbers above describe 2024. But capacity planning looks forward, and the explainer warned that forecasts assume the future will resemble the past. What if solar capacity grows by 20%? What if demand rises by 5% as heating electrifies? Scenario analysis tests how sensitive the recommendation is to changes in assumptions. It is the bridge between prediction and prescription.

In [ ]:
def analyse_scenario(demand_series, solar_series, wind_series, threshold, label):
    """Calculate critical hours and worst-case shortfall for a scenario."""
    renewable = solar_series + wind_series
    gap = demand_series - renewable
    shortfall = (gap - threshold).clip(lower=0)
    
    is_critical = shortfall > 0
    critical_count = is_critical.sum()
    
    # Find worst consecutive block
    blocks = (is_critical != is_critical.shift()).cumsum()
    block_sums = shortfall[is_critical].groupby(blocks[is_critical]).sum()
    worst_mwh = block_sums.max() if len(block_sums) > 0 else 0
    
    return {
        "scenario": label,
        "critical_hours": int(critical_count),
        "pct_critical": round(critical_count / len(demand_series) * 100, 1),
        "worst_event_mwh": round(float(worst_mwh), 0),
        "worst_event_gwh": round(float(worst_mwh) / 1000, 1),
    }

In [ ]:
scenarios = [
    analyse_scenario(hourly["demand"], hourly["solar"], hourly["wind"],
                     threshold, "Baseline (2024)"),
    analyse_scenario(hourly["demand"], hourly["solar"] * 1.20, hourly["wind"],
                     threshold, "Solar +20%"),
    analyse_scenario(hourly["demand"], hourly["solar"] * 1.20, hourly["wind"] * 1.10,
                     threshold, "Solar +20%, Wind +10%"),
    analyse_scenario(hourly["demand"] * 1.05, hourly["solar"], hourly["wind"],
                     threshold, "Demand +5%"),
    analyse_scenario(hourly["demand"] * 1.05, hourly["solar"] * 1.20, hourly["wind"] * 1.10,
                     threshold, "Demand +5%, Solar +20%, Wind +10%"),
]

scenario_df = pd.DataFrame(scenarios)
scenario_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(scenario_df["scenario"], scenario_df["critical_hours"])
axes[0].set_title("Critical Hours per Scenario")
axes[0].set_xlabel("Hours")

axes[1].barh(scenario_df["scenario"], scenario_df["worst_event_gwh"])
axes[1].set_title("Worst-Case Storage Needed (GWh)")
axes[1].set_xlabel("GWh")

plt.tight_layout()
plt.show()

---

## 6. Stakeholder briefing

The analysis is complete. The final step is to communicate it in the structure the explainer recommended: separate what the data shows from what we predict from what we recommend. The reader should always know where the evidence ends and the judgement begins.

The following three sections model the observation/prediction/recommendation format. Notice how each section uses different language and carries a different level of confidence.

### Observation

*This section is descriptive: it reports what the data shows, in the past tense, with no predictions or recommendations.*

In 2024, the grid experienced a significant supply gap between total demand and renewable generation (solar + wind). The gap was concentrated in winter months (November through February) and during evening hours (17:00 to 22:00), when demand for heating and lighting peaked while solar output was zero.

Solar and demand operate on opposite seasonal cycles: solar peaks in June/July while demand peaks in January/December. Wind partially offsets this mismatch with higher winter output, but not by enough to close the gap.

### Prediction

*This section is predictive: it estimates what will happen, uses conditional language, and acknowledges uncertainty.*

Based on rolling average forecasts validated against December 2024 actuals:

- Under baseline conditions, the supply gap exceeds 30,000 MW for a significant proportion of winter hours.
- Increasing solar capacity by 20% reduces critical hours but does not eliminate them, because the worst gaps occur at night when solar output is zero regardless of installed capacity.
- A 5% increase in demand (e.g. from electrification of heating) worsens the gap substantially.
- The combination of demand growth and renewable expansion still leaves a meaningful shortfall in winter evenings.

These forecasts assume that 2024 patterns are representative of future years. A structural break (the explainer's term for a sudden, permanent change in behaviour) would invalidate these projections.

### Recommendation

*This section is prescriptive: it recommends specific actions, states the assumptions behind each one, and notes what would change the recommendation.*

1. **Build battery storage** sized to cover the worst-case consecutive shortfall event identified in the scenario analysis. The baseline scenario requires the capacity shown in the table above; planning should use the demand +5% scenario as the prudent sizing target.

2. **Prioritise wind over solar** for near-term capacity expansion. Wind generates during winter evenings when the gap is largest. Solar expansion helps with annual totals but does not address the critical winter evening shortfall.

3. **Invest in demand-side response** programmes to reduce the evening peak. Shifting 5-10% of evening demand to off-peak hours (e.g. pre-heating buildings in the afternoon, smart appliance scheduling) would reduce the required battery size significantly.

4. **Revisit these forecasts quarterly** as actual 2025 data becomes available. The simple rolling average forecasts used here have known limitations (they do not account for weather forecasts or temperature-demand correlation). More sophisticated methods should be evaluated once the business case for battery investment is confirmed.

*If gas prices fall significantly, gas generation may become cheaper than battery storage and recommendation 1 would need to be re-evaluated. If government policy accelerates wind deployment beyond the +10% scenario, the required battery capacity shrinks.*

---

## Exercises

We have now seen the full pipeline: from raw data to descriptive analysis, through prediction, to a prescriptive recommendation. These exercises give you practice with different thresholds, scenarios, and the writing itself.

### Exercise 1: Different threshold

Re-run the critical hours analysis with a threshold of 25,000 MW instead of 30,000 MW. How many additional critical hours does the lower threshold produce? Plot the monthly distribution of critical hours for both thresholds side by side.

In [ ]:
# Your code here


### Exercise 2: Custom scenario

Create a scenario where solar capacity doubles (+100%) and wind grows by 30%, but demand also grows by 10% due to electric vehicle charging. Use the `analyse_scenario` function. How does this compare to the baseline? Write a one-paragraph interpretation of the result.

In [ ]:
# Your code here


### Exercise 3: Surplus energy analysis

When renewable output exceeds demand, there is a surplus that could charge batteries. Calculate the total surplus energy (in GWh) across 2024. In which months does the surplus occur? Is the annual surplus enough to cover the annual shortfall? What does this tell you about the feasibility of seasonal storage?

In [ ]:
# Your code here


### Exercise 4: Write your own briefing

Choose the scenario from Exercise 2 (or create a different one). Write a stakeholder briefing in three markdown cells below, following the observation/prediction/recommendation structure from Section 6. Use specific numbers from your analysis. Keep each section to 3-5 sentences.

In [ ]:
# Your code here


---

## Summary

This notebook covered the final step in the analysis pipeline: turning predictions into decisions.

- **Combining forecasts** for demand, solar, and wind into a single supply gap metric gives the operator one number to plan around.
- **Critical hours analysis** identifies when and how often the gap exceeds a threshold, revealing the timing and severity of the problem.
- **Battery sizing** from worst-case consecutive shortfall events translates a statistical finding into a concrete engineering requirement.
- **Scenario analysis** tests how sensitive the recommendation is to changes in assumptions (demand growth, renewable expansion). It makes the boundary conditions of our advice explicit.
- **The stakeholder briefing** separates observation (what happened), prediction (what we expect), and recommendation (what to do). This is the communication framework the explainer described. It prevents the common mistake of mixing facts with judgements, and it lets the reader assess each level of confidence independently.

The three notebooks together trace the full arc of a time series analysis: from exploring the raw patterns, through building and evaluating forecasts, to translating those forecasts into actionable recommendations with visible assumptions.